In [45]:
from fastai.torch_basics import *
from fastai.data.all import *
from fastai.text.data import *
from tokenizers import Tokenizer
from datasets import load_dataset

In [46]:
#| default_exp text.tokenizer


# Text tokenizer

> Encode texts with a Hugging Face tokenizer into a flat cache of token IDs and offsets for each document

In [47]:
EOS_TOK = "<eos>"
BOS_TOK = "<bos>"
PAD_TOK = "<pad>"

def hf_tokenizer(name='gpt2'):
    "Pretrained Hugging Face tokenizer `name`, with `EOS_TOK`, `BOS_TOK` and `PAD_TOK` added as special tokens"
    tok = Tokenizer.from_pretrained(name)
    tok.add_special_tokens([EOS_TOK, BOS_TOK, PAD_TOK])
    return tok

In [48]:
tok = hf_tokenizer()
pad_id = tok.token_to_id(PAD_TOK)
tok.get_vocab_size(), pad_id

(50260, 50259)

## Token cache

A cache is a directory holding the IDs, the offsets and the tokenizer that produced them, so it can be decoded later without knowing how it was built. 

In [49]:
def save_token_cache(path, flat, offsets, tok, metadata):
    "Save token IDs, offsets, tokenizer and document metadata to `path`"
    if len(metadata) != len(offsets)-1: raise ValueError('metadata rows must match the number of documents')
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    np.save(path/'flat.npy', flat)
    np.save(path/'offsets.npy', offsets)
    metadata.to_csv(path/'metadata.csv', index=False)
    tok.save(str(path/'tokenizer.json'))

def load_token_cache(path):
    "Load token IDs, offsets, tokenizer and document metadata from `path`"
    path = Path(path)
    flat, offsets = np.load(path/'flat.npy'), np.load(path/'offsets.npy')
    tok = Tokenizer.from_file(str(path/'tokenizer.json'))
    metadata = pd.read_csv(path/'metadata.csv', dtype=str)
    return flat, offsets, tok, metadata

In [50]:
cache_flat = np.array([1, 2, 3], dtype=np.uint16)
cache_offsets = np.array([0, 2, 3], dtype=np.int64)
with tempfile.TemporaryDirectory() as d:
    metadata = pd.DataFrame(dict(path=['train/pos/1.txt', 'test/neg/2.txt'], label=['pos', 'neg'], split=['train', 'test']))
    save_token_cache(d, cache_flat, cache_offsets, tok, metadata)
    flat2, offsets2, tok2, metadata2 = load_token_cache(d)
test_eq(flat2, cache_flat)
test_eq(offsets2, cache_offsets)
test_eq(metadata2.to_dict('list'), metadata.to_dict('list'))
metadata2

,path,label,split
0,train/pos/1.txt,pos,train
1,test/neg/2.txt,neg,test


## Encoding texts

`tokenize_texts` encodes a collection of texts with the fast batch encoder and returns two arrays: `flat`, every document's token IDs, and `offsets`, where document `i` is `flat[offsets[i]:offsets[i+1]]`

In [51]:
def id_dtype(tok):
    "The smallest type that holds every ID of `tok`"
    return np.uint16 if tok.get_vocab_size(with_added_tokens=True) <= 2**16 else np.uint32

In [52]:
def tokenize_texts(
    items, # Texts, or the items `get_texts` extracts them from
    tok, # HF `Tokenizer`
    get_texts=None, # Column of texts from Dataset, or function from `items` to the texts
    bs=1024, 
):
    "Encode texts into flat token IDs and document offsets, where document `i` is `flat[offsets[i]:offsets[i+1]]`"
    if isinstance(get_texts, str): get_texts = itemgetter(get_texts)
    texts = items if get_texts is None else get_texts(items)
    texts = list(texts)
    dtype = id_dtype(tok)
    chunks = [np.asarray(e.ids, dtype=dtype) for i in progress_bar(range(0, len(texts), bs)) for e in tok.encode_batch_fast(texts[i:i+bs])]
    flat = np.concatenate(chunks) if chunks else np.empty(0, dtype=dtype)
    offsets = np.cumsum([0] + [len(c) for c in chunks], dtype=np.int64)
    return flat, offsets

In [53]:
texts = ["The quick brown fox", "jumps over the lazy dog"]
flat, offsets = tokenize_texts(texts, tok)
test_eq(offsets[-1], len(flat))
flat, offsets

 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

(array([  464,  2068,  7586, 21831,    73,  8142,   625,   262, 16931,
         3290], dtype=uint16),
 array([ 0,  4, 10]))

In [54]:
flat

array([  464,  2068,  7586, 21831,    73,  8142,   625,   262, 16931,
        3290], dtype=uint16)

In [55]:
offsets

array([ 0,  4, 10])

When the items are a table, a `Dataset` or a `DataFrame`, `get_texts` names the column to encode. A function from the items to their texts covers every other case.


In [56]:
df = pd.DataFrame({'id': range(len(texts)), 'text': texts})
flat_df, offsets_df = tokenize_texts(df, tok, get_texts='text')
test_eq(flat_df, flat)
test_eq(offsets_df, offsets)


 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

`split_docs` splits the flat array back into one array per document.

In [57]:
def split_docs(flat, offsets):
    "The documents of a token cache, as views into `flat`"
    return L(flat[i:j] for i,j in zip(offsets[:-1], offsets[1:]))

In [58]:
docs = split_docs(flat, offsets)
test_eq(len(docs), len(texts))
tok.decode(docs[1].tolist())

'jumps over the lazy dog'

## Hugging Face datasets

To build a cache, supply either `text_col` or a custom `get_data` callable. For one document per source row, `text_col` selects the texts and `metadata_cols` selects the metadata columns. The source split is added automatically. For custom processing, `get_data` returns `(texts, metadata)`, with one metadata row per final document and no text in the DataFrame. WikiText-103 uses this callable to assemble source lines into articles.


`split=None` includes every source split. Pass a list such as `['train', 'validation']` to select several, or a string for one split. `get_dataset_data` processes each selected split separately and returns combined texts and metadata. Article assembly stays within each split. `tokenize_dataset` handles tokenization, saving and loading the cache. Default cache paths distinguish split selections; caches without `metadata.csv` are rebuilt.

In [59]:
def get_dataset_data(
    name, # Hugging Face Hub dataset name or local dataset directory
    get_data=None, # Optional callable returning (texts, metadata DataFrame) for each split
    subset=None,
    split=None, # All splits if None, or a split name/expression or list of splits
    text_col=None, # Text column; required when get_data is not supplied
    metadata_cols=(), # Columns to cache alongside the source split
):
    "Load selected dataset splits and combine their prepared texts and metadata"
    if get_data is None and text_col is None: raise ValueError('Provide text_col or get_data')
    dsets = load_dataset(name, subset, split=split)
    if split is None: dsets = dsets.values()
    elif isinstance(split, str): dsets = [dsets]
    texts, metadata = [], []
    for ds in dsets:
        if get_data is None:
            split_texts = ds[text_col]
            split_metadata = pd.DataFrame({c: ds[c] for c in metadata_cols}, index=range(len(ds)))
            split_metadata['split'] = str(ds.split)
        else: split_texts, split_metadata = get_data(ds)
        texts.extend(split_texts)
        metadata.append(split_metadata)
    return texts, pd.concat(metadata, ignore_index=True)

In [60]:
@delegates(get_dataset_data)
def tokenize_dataset(
    name, # Hugging Face Hub dataset name or local dataset directory
    subset=None,
    split=None, # All splits if None, or a split name/expression or list of splits
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    cache_dir=None, # Cache directory; defaults to a path keyed by source, selected splits and tokenizer
    bs=1024, # Texts per tokenizer batch
    **kwargs
):
    "Build or load a token cache for a Hugging Face Hub dataset or local dataset directory"
    split_key = 'all' if split is None else str(split)
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/name/(subset or 'default')/split_key/tok
    if not (Path(cache_dir)/'metadata.csv').exists():
        texts, metadata = get_dataset_data(name, subset=subset, split=split, **kwargs)
        tok = hf_tokenizer(tok)
        flat, offsets = tokenize_texts(texts, tok, bs=bs)
        save_token_cache(cache_dir, flat, offsets, tok, metadata)
    return load_token_cache(cache_dir)


The following examples share a tiny local dataset containing two reviews in each source split.

In [61]:
from contextlib import contextmanager

In [62]:
@contextmanager
def sample_dataset():
    with tempfile.TemporaryDirectory() as d:
        source = Path(d)/'source'
        source.mkdir()
        for split_name in ['train', 'validation', 'test']:
            (source/f'{split_name}.csv').write_text(r"""text,label
Great film,pos
Awful film,neg
""")
        yield str(source), Path(d)/'cache'

In [63]:
def get_sample_data(ds): return ds['text'], pd.DataFrame(dict(label=ds['label'], split=str(ds.split)))
with sample_dataset() as (source, _):
    _, all_metadata = get_dataset_data(source, text_col='text')
    test_eq(set(all_metadata.split), {'train', 'validation', 'test'})
    _, train_metadata = get_dataset_data(source, get_data=get_sample_data, split='train')
    test_eq(train_metadata.split.tolist(), ['train', 'train'])
train_metadata

Generating train split: 2 examples [00:00, 1693.64 examples/s]
Generating validation split: 2 examples [00:00, 2127.47 examples/s]
Generating test split: 2 examples [00:00, 2525.17 examples/s]


,label,split
0,pos,train
1,neg,train


Select training and validation reviews by column name, then inspect their cached labels and source splits.

In [64]:
with sample_dataset() as (source, cache_dir):
    _, _, _, review_metadata = tokenize_dataset(source, text_col='text', metadata_cols=['label'],
        split=['train', 'validation'], cache_dir=cache_dir)
    test_eq(review_metadata.split.tolist(), ['train', 'train', 'validation', 'validation'])
    test_eq(review_metadata.label.tolist(), ['pos', 'neg']*2)
review_metadata

Generating train split: 2 examples [00:00, 1573.55 examples/s]
Generating validation split: 2 examples [00:00, 1597.53 examples/s]
Generating test split: 2 examples [00:00, 1833.17 examples/s]


 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

,label,split
0,pos,train
1,neg,train
2,pos,validation
3,neg,validation


Supply either a text column or a custom data callable when preparing a dataset.

In [65]:
test_fail(lambda: get_dataset_data('stanfordnlp/imdb'), exc=ValueError, contains='Provide text_col or get_data')

WikiText-103 stores article text across multiple source rows, with headings marking article boundaries. `get_wiki103_data` joins each article's rows into one document and returns one metadata row with the source split.

In [66]:
_wiki_heading = re.compile(r'^ = [^=].* = \n$')

def get_wiki103_data(ds):
    "Assemble WikiText-103 articles and their metadata from source lines"
    docs, cur = [], []
    for text in ds['text']:
        if _wiki_heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    if cur: docs.append(''.join(cur))
    texts = [d for d in docs if d.strip()]
    return texts, pd.DataFrame(dict(split=[str(ds.split)]*len(texts)))

In [67]:
#| eval: false
flat_wiki, offsets_wiki, tok_wiki, metadata_wiki = tokenize_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, split='validation')
len(offsets_wiki)-1, len(flat_wiki)


(60, 247289)

In [68]:
#| eval: false
sorted(p.name for p in (fastai_path('data')/'token_cache/Salesforce/wikitext/wikitext-103-raw-v1/validation/gpt2').iterdir())

['flat.npy', 'metadata.csv', 'offsets.npy', 'tokenizer.json']

In [69]:
#| eval: false
tok_wiki.decode(split_docs(flat_wiki, offsets_wiki)[0].tolist())[:80]


' = Homarus gammarus = \n Homarus gammarus , known as the European lobster or comm'

Simple English Wikipedia has one article per row in its `md` column. Pass `text_col='md'` to tokenize those articles. `tokenize_dataset` adds each article's source split to the metadata.

In [70]:
#| eval: false
flat_sw, offsets_sw, _, metadata_sw = tokenize_dataset('answerdotai/simplewiki', subset='articles',
    text_col='md', split='train[:500]')
len(offsets_sw)-1, len(flat_sw)


(500, 752440)

In [71]:
#| eval: false
tok.decode(split_docs(flat_sw, offsets_sw)[0].tolist())[:80]


'**April** (Apr.) is the fourth month of the year in the Julian and Gregorian cal'

## Text files

`get_folder_data` lists files under `path` in sorted order and prepares their metadata. The default uses the standard `split/label/file` layout to record each relative path, label, and source split. Pass `get_metadata` for other layouts; it receives the sorted relative paths and returns one row per file. `tokenize_folder` reuses the saved tokens and metadata when the ordered paths match. Changed paths rebuild the cache. To change labels or splits for the same paths, rebuild the cache by deleting it or choosing a new `cache_dir`. The function returns the files, their tokenized documents, the tokenizer, and metadata in matching order.

In [72]:
def get_folder_data(
    path, # Folder of text files
    get_metadata=None # Callable returning metadata columns for nonstandard folder layouts
):
    "List sorted text files under `path` and prepare their metadata"
    path = Path(path)
    files = get_text_files(path).sorted()
    rel_files = [f.relative_to(path) for f in files]
    paths = [str(f) for f in rel_files]
    if get_metadata is None:
        metadata = pd.DataFrame(dict(
            path=paths,
            label=[f.parent.name if len(f.parts) >= 2 else None for f in rel_files],
            split=[f.parents[1].name if len(f.parts) >= 3 else None for f in rel_files]))
    else:
        metadata = get_metadata(rel_files)
        if len(metadata) != len(files): raise ValueError('get_metadata must return one row per file')
        metadata = pd.concat([pd.DataFrame(dict(path=paths)), metadata.reset_index(drop=True)], axis=1)
    return files, metadata

In [73]:
@delegates(tokenize_texts, but=['get_texts'])
def tokenize_folder(
    path, # Folder of text files
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    get_metadata=None, # Callable returning metadata columns for nonstandard folder layouts
    cache_dir=None, # Cache directory; `fastai_path('data')/'token_cache'/path.name/tok` if None
    **kwargs
):
    "Cache text files under `path` with paths and metadata"
    path = Path(path)
    files, metadata = get_folder_data(path, get_metadata)
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/path.name/tok
    cache_meta = Path(cache_dir)/'metadata.csv'
    if cache_meta.exists():
        flat, offsets, tokenizer, cached_metadata = load_token_cache(cache_dir)
        if 'path' in cached_metadata and metadata['path'].tolist() == cached_metadata['path'].tolist():
            docs = split_docs(flat, offsets)
            if len(docs) != len(files): raise ValueError(f'{cache_dir} holds {len(docs)} documents for {len(files)} files: delete it to rebuild')
            return files, docs, tokenizer, cached_metadata
    tokenizer = hf_tokenizer(tok)
    flat, offsets = tokenize_texts(files, tokenizer,
        get_texts=lambda fs: [f.read_text(encoding='utf-8') for f in fs], **kwargs)
    save_token_cache(cache_dir, flat, offsets, tokenizer, metadata)
    return files, split_docs(flat, offsets), tokenizer, pd.read_csv(cache_meta, dtype=str)

In [74]:
#| eval: false
path = untar_data(URLs.IMDB)
files, imdb_docs, tok_imdb, imdb_metadata = tokenize_folder(path)
len(files), files[0].relative_to(path), imdb_metadata.iloc[0], tok_imdb.decode(imdb_docs[0][:30].tolist())

(100002,
 Path('test/neg/0_2.txt'),
 path     test/neg/0_2.txt
 label                 neg
 split                test
 Name: 0, dtype: str,
 'Once again Mr. Costner has dragged out a movie for far longer than necessary. Aside from the terrific sea rescue sequences, of which there are very')

## Transforms

`HfTokenizer` turns a cached document, or a raw string, into a `TensorText` with `<bos>` and `<eos>` around it, and decodes a `TensorText` back to a string. Its `vocab` and `pad_id` are what `language_model_learner` and `text_classifier_learner` read from `dls`, through the pipeline. `get_lengths` gives the length of each document once the boundary tokens are added, which `LMDataLoader` and `SortedDL` ask for the same way. An empty collection returns an empty length list. It returns `None` for items that are neither arrays nor strings, such as the `(doc, label)` pairs of a classifier, and the loader then measures them itself. `Reverse` flips a sequence in both directions. Placed after `HfTokenizer`, it is all a backward language model needs: the loaders, the model and the classifier stay the same.

In [75]:
BOS_TOK, EOS_TOK, PAD_TOK

('<bos>', '<eos>', '<pad>')

In [76]:
class HfTokenizer(Transform):
    "Encode texts or cached token IDs with `tok`, adding `BOS_TOK` and `EOS_TOK`, and decode them back to text"
    input_types = (str, np.ndarray)
    def __init__(self, tok):
        self.tok = tok
        self.bos,self.eos,self.pad_id = (tok.token_to_id(t) for t in (BOS_TOK, EOS_TOK, PAD_TOK))
        self.vocab = [tok.id_to_token(i) for i in range(tok.get_vocab_size())]
    def _add_boundaries(self, ids): return TensorText([self.bos] + ids + [self.eos])
    def _len(self, o): return len(o) if isinstance(o, np.ndarray) else len(self.tok.encode(o).ids)
    def get_lengths(self, items):
        items = L(items)
        if not items: return L()
        return L(self._len(o)+2 for o in items) if isinstance(items[0], (np.ndarray, str)) else None
    def encodes(self, s:str): return self._add_boundaries(self.tok.encode(s).ids)
    def encodes(self, s:np.ndarray): return self._add_boundaries(s.tolist())
    def decodes(self, t:TensorText): return TitledStr(self.tok.decode(t.tolist(), skip_special_tokens=False))

In [77]:
tfm = HfTokenizer(tok)
t = tfm(docs[0])
test_eq(tfm.get_lengths(texts), tfm.get_lengths(docs))
test_eq(tfm.get_lengths([]), [])
t, tfm.decode(t), tfm.get_lengths(docs)

(TensorText([50258,   464,  2068,  7586, 21831, 50257]),
 '<bos>The quick brown fox<eos>',
 [6, 8])

A raw string is encoded first, then given the same boundaries, so a text and its cached document give the same `TensorText`.

In [78]:
test_eq(tfm(texts[0]), t)
tfm(texts[1]), tfm.decode(tfm(texts[1]))

(TensorText([50258,    73,  8142,   625,   262, 16931,  3290, 50257]),
 '<bos>jumps over the lazy dog<eos>')

In [79]:
class Reverse(Transform):
    def encodes(self, o): return o.flip(0)
    def decodes(self, o): return o.flip(0)


In [80]:
pipe = Pipeline([tfm, Reverse()])
test_eq(pipe(docs[0]), t.flip(0))
pipe.decode(pipe(docs[0]))


'<bos>The quick brown fox<eos>'

## DataLoaders

`HfTextBlock` is this module's `TextBlock`. The pipeline is `tok_tfm` alone, with `Reverse` after it for a backward model, because the tokenizer already produces ids and there is no `Numericalize` step. Padding for classification uses the tokenizer's `pad_id`.

In [81]:
class HfTextBlock(TransformBlock):
    "A `TransformBlock` for documents that `tok_tfm` encodes straight to token ids"
    def __init__(self, tok_tfm, is_lm=False, seq_len=72, backwards=False):
        type_tfms = [tok_tfm] + ([Reverse()] if backwards else [])
        super().__init__(type_tfms=type_tfms, dl_type=LMDataLoader if is_lm else SortedDL,
                         dls_kwargs={'seq_len': seq_len} if is_lm else {'before_batch': Pad_Chunk(seq_len=seq_len, pad_idx=tok_tfm.pad_id)})

In [82]:
blk = HfTextBlock(tfm, is_lm=True, seq_len=3)
dls = DataBlock(blocks=blk, splitter=RandomSplitter(0.5, seed=42)).dataloaders(docs, bs=1)
x, y = dls.one_batch()
x.shape, len(dls.vocab), dls.pad_id

(torch.Size([1, 3]), 50260, 50259)

`TextDataLoadersNew` will replace `TextDataLoaders` when the migration is complete. `from_dataset` builds a language model by default. Pass `get_y='label'` to classify using the cached `label` column. For the column-based preparation path, that column is added to `metadata_cols` automatically. A custom `get_data` callable must include it in its returned metadata. Pass `text_col` or `get_data` through to `tokenize_dataset` as before.

`get_metadata_splits` returns row positions in the original document order, keeping tokens and labels aligned. With `valid_pct=None`, `from_dataset` requests `split=[train, valid]` automatically unless `split` is supplied explicitly. The helper selects those named splits from the cached metadata. With `valid_pct`, it creates a new random split from all selected rows, ignoring saved split labels. Use `split` to choose the source data for random splitting, such as `split=['train', 'validation']` to leave out test data. Named splits match metadata values exactly, including slice expressions. Rebuild an existing cache if it lacks a required label column, or a split column needed for named splitting.

`from_folder` reads labels and splits from the cache returned by `tokenize_folder`. It defaults to classification using `label`, with `train` and `valid` as the source split names. Set `get_y` for another label column or `get_metadata` to prepare metadata for a custom layout. `folders` selects top-level directories using the cached relative paths. With `valid_pct`, all rows from that selection are combined before random splitting. Without `folders`, the random split uses all cached rows.

In [83]:
def get_metadata_splits(
    metadata, # One metadata row per cached document
    train='train', # Training split name
    valid='validation', # Validation split name
    valid_pct=None, # Random validation fraction; None uses named source splits
    seed=None, # Random split seed
    folders=None # Top-level source folders to include; valid_pct splits all their rows
):
    "Training and validation row positions from cached metadata"
    rows = L(range_of(metadata))
    if folders is not None:
        folders = L(folders)
        rows = L(mask2idxs(metadata['path'].map(lambda p: Path(p).parts[0] in folders)))
        metadata = metadata.iloc[rows]
    if valid_pct is not None:
        train_idx, valid_idx = RandomSplitter(valid_pct, seed=seed)(rows)
        return rows[train_idx], rows[valid_idx]
    return [rows[mask2idxs(metadata['split'] == name)] for name in (train, valid)]

Like Fastai's splitters, `get_metadata_splits` returns an empty index list when a requested split has no rows.

In [84]:
split_metadata = pd.DataFrame(dict(split=['train']))
split_indices = get_metadata_splits(split_metadata)
test_eq(split_indices, ([0], []))
test_eq(get_metadata_splits(split_metadata.iloc[:0]), ([], []))
split_indices

[[0], []]

In [85]:
class TextDataLoadersNew(DataLoaders):
    "Wrapper around several `DataLoader`s with factory methods for texts encoded by `HfTokenizer`"
    @classmethod
    @delegates(tokenize_dataset)
    def from_dataset(
        cls,
        name, # Hub dataset name or local dataset directory
        get_y=None, # Label column in cached metadata; None for a language model
        train='train', # Source split used for training
        valid='validation', # Source split used for validation
        valid_pct=None, # Random validation fraction of selected rows; None uses source splits
        metadata_cols=(), # Additional source columns to cache; get_y is included automatically
        seed=None, # Random split seed
        seq_len=72, # Language model sequence length or classifier padding chunk size
        backwards=False, # Reverse each tokenized document
        bs=64, # DataLoader batch size
        **kwargs
    ):
        "Build language model or classification loaders using cached document metadata"
        metadata_cols = (L(metadata_cols) + L(get_y)).unique()
        if valid_pct is None: kwargs.setdefault('split', [train, valid])
        flat, offsets, tok, metadata = tokenize_dataset(name, metadata_cols=metadata_cols, **kwargs)
        docs = split_docs(flat, offsets)
        splits = get_metadata_splits(metadata, train=train, valid=valid, valid_pct=valid_pct, seed=seed)
        is_lm = get_y is None
        blocks = [HfTextBlock(HfTokenizer(tok), is_lm=is_lm, seq_len=seq_len, backwards=backwards)]
        if not is_lm: blocks.append(CategoryBlock)
        dblock = DataBlock(blocks=blocks, get_x=lambda i: docs[i],
            get_y=None if is_lm else (lambda i: metadata[get_y].iloc[i]), splitter=lambda _: splits)
        return cls.from_dblock(dblock, range_of(docs), bs=bs, seq_len=seq_len)

    @classmethod
    @delegates(DataLoaders.from_dblock)
    def from_folder(
        cls,
        path, # Folder of text files
        train='train', # Training split name in cached metadata
        valid='valid', # Validation split name in cached metadata
        valid_pct=None, # Random validation fraction; None uses source splits
        seed=None, # Random split seed
        is_lm=False, # Build a language model instead of a classifier
        tok='gpt2', # Tokenizer name for hf_tokenizer
        seq_len=72, # Language model sequence length or classifier padding chunk size
        backwards=False, # Reverse each tokenized document
        folders=None, # Source folders to include; with valid_pct, combine their rows before splitting
        cache_dir=None, # Token cache directory
        get_y='label', # Cached label column for classification
        get_metadata=None, # Callable returning metadata columns for nonstandard folder layouts
        **kwargs
    ):
        "Build language model or classification loaders from a folder's cached metadata"
        _, docs, tok, metadata = tokenize_folder(path, tok, get_metadata=get_metadata, cache_dir=cache_dir)
        splits = get_metadata_splits(metadata, train=train, valid=valid, valid_pct=valid_pct, seed=seed, folders=folders)
        blocks = [HfTextBlock(HfTokenizer(tok), is_lm=is_lm, seq_len=seq_len, backwards=backwards)]
        if not is_lm: blocks.append(CategoryBlock)
        dblock = DataBlock(blocks=blocks, get_x=lambda i: docs[i],
            get_y=None if is_lm else (lambda i: metadata[get_y].iloc[i]), splitter=lambda _: splits)
        return cls.from_dblock(dblock, range_of(docs), path=path, seq_len=seq_len, **kwargs)


The local dataset has training, validation, and test splits. The classifier uses cached labels and selects the training and validation rows. Moving the source directory before the second call verifies that the loaders can be built from the cache alone.

In [86]:
with sample_dataset() as (source, cache_dir):
    dls = TextDataLoadersNew.from_dataset(source, text_col='text', get_y='label', cache_dir=cache_dir, bs=2)
    cached_metadata = pd.read_csv(cache_dir/'metadata.csv')
    test_eq(set(cached_metadata['split']), {'train', 'validation'})
    test_eq(cached_metadata.iloc[dls.train_ds.items]['split'].tolist(), ['train', 'train'])
    test_eq(cached_metadata.iloc[dls.valid_ds.items]['split'].tolist(), ['validation', 'validation'])
    test_eq(dls.vocab[1], ['neg', 'pos'])
    batch = dls.one_batch()
batch

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

Generating train split: 2 examples [00:00, 1586.05 examples/s]
Generating validation split: 2 examples [00:00, 3264.05 examples/s]
Generating test split: 2 examples [00:00, 4328.49 examples/s]


(TensorText([[50258, 23155,   913,  2646, 50257],
             [50258, 13681,  2646, 50257, 50259]], device='mps:0'),
 TensorCategory([0, 1], device='mps:0'))

In [ ]:
#| eval: false
dls = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:1%]', bs=8)
dls.show_batch(max_n=2)

<bos> = Exit Through the Kwik @-@ E @-@ Mart = 
 " Exit Through the Kwik @-@ E @-@ Mart " is the fifteenth episode of the twenty @-@ third season of the American animated television sitcom The Simpsons . It originally aired on the Fox network in the United States on March 4 , 2012 .
 of classic movies . 
 Highland Park houses a Jimmie Rodgers museum which displays the original guitar of " The Singing Brakeman " and other memorabilia of his life and career , as well as railroad equipment from the steam @-@ engine era . In addition to the museum building itself , there are outside memorials , and a vintage steam locomotive
 = Exit Through the Kwik @-@ E @-@ Mart = 
 " Exit Through the Kwik @-@ E @-@ Mart " is the fifteenth episode of the twenty @-@ third season of the American animated television sitcom The Simpsons . It originally aired on the Fox network in the United States on March 4 , 2012 . In
 classic movies . 
 Highland Park houses a Jimmie Rodgers museum which displays the origi

In [ ]:
#| eval: false
dls = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', get_y='label',
    split='train', valid_pct=0.2, seed=42, bs=8)
dls.show_batch(max_n=2)


,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>If you've ever been harassed on the Underground by a Christian who says, ""Jesus is the answer. What's the question?"", then perhaps you should thank God if you've never met a Lacanian. Slavoj Zizek, the most evangelical of Lacanians, would surely exchange the word ""Jesus"" in that statement for ""Lacan/Hegel"".<br /><br />Zizek's star burns brightly at the moment, no doubt because we generally view films and pop culture purely as entertainment for our consumption. So it seems impressive when someone - anyone - comes along and says, ""Hang on, films may say something about ourselves.""<br /><br />The ideas Zizek expounds in this film are ""true"" purely because he says so. For example, Zizek explains that three Marx Bros are the ego, superego and id (God knows what happened to Zeppo, or Gummo perhaps they're the sinthome...or is that movies themselves?). This is simply what they are. In Zizek's output,",neg


In [ ]:
@contextmanager
def sample_text_folder():
    with tempfile.TemporaryDirectory() as d:
        source = Path(d)/'texts'
        for split_name in ['train', 'valid', 'test']:
            for label, text in [('neg', 'Awful film'), ('pos', 'Great film')]:
                file = source/split_name/label/'review.txt'
                file.parent.mkdir(parents=True, exist_ok=True)
                file.write_text(text, encoding='utf-8')
        for folder in ['unsup', 'ignored']:
            (source/folder).mkdir()
            (source/folder/'review.txt').write_text('An unlabeled film review', encoding='utf-8')
        yield source, Path(d)/'cache'

A custom metadata function can define labels and splits independently of the folder names. These local files use the standard folder layout, but their metadata names the splits `fit` and `holdout` and stores labels in `sentiment`.

In [ ]:
def custom_folder_metadata(files):
    labels, splits = {'neg': 'negative', 'pos': 'positive'}, {'train': 'fit', 'valid': 'holdout'}
    return pd.DataFrame(dict(sentiment=[labels.get(f.parent.name) for f in files],
        split=[splits.get(f.parts[0], f.parts[0]) for f in files]))
with sample_text_folder() as (source, cache_dir):
    folder_dls = TextDataLoadersNew.from_folder(source, train='fit', valid='holdout', get_y='sentiment',
        get_metadata=custom_folder_metadata, cache_dir=cache_dir, bs=2)
    cached_metadata = pd.read_csv(cache_dir/'metadata.csv')
    test_eq(cached_metadata.iloc[folder_dls.train_ds.items]['split'].unique().tolist(), ['fit'])
    test_eq(cached_metadata.iloc[folder_dls.valid_ds.items]['split'].unique().tolist(), ['holdout'])
    test_eq(folder_dls.vocab[1], ['negative', 'positive'])
    folder_batch = folder_dls.one_batch()
folder_batch

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

(TensorText([[50258, 23155,   913,  2646, 50257],
             [50258, 13681,  2646, 50257, 50259]], device='mps:0'),
 TensorCategory([0, 1], device='mps:0'))

For a language model, explicitly list the folders whose documents should be combined before a random split. Here, training, test, and unsupervised files are included; files from the `valid` folder are excluded.

In [ ]:
with sample_text_folder() as (source, cache_dir):
    folder_lm = TextDataLoadersNew.from_folder(source, is_lm=True, valid_pct=0.4, seed=42,
        folders=['train', 'test', 'unsup'], cache_dir=cache_dir, bs=1, seq_len=2)
    cached_metadata = pd.read_csv(cache_dir/'metadata.csv')
    selected = cached_metadata['path'].map(lambda p: Path(p).parts[0] in ['train', 'test', 'unsup'])
    used = list(folder_lm.train_ds.items) + list(folder_lm.valid_ds.items)
    test_eq(sorted(used), mask2idxs(selected))
    x, y = folder_lm.one_batch()
    test_eq(x[:, 1:], y[:, :-1])
x, y

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

(LMTensorText([[50258, 13681]], device='mps:0'),
 TensorText([[13681,  2646]], device='mps:0'))

## IMDb

The IMDb reviews come as text files in `train` and `test` folders, with `pos`, `neg` and `unsup` subfolders. `from_folder` with `valid='test'` gives the classifier loaders over the labelled reviews, and with `is_lm=True` and `valid_pct` the language model loaders over all of them.


In [ ]:
#| eval: false
dls = TextDataLoadersNew.from_folder(path, valid='test', bs=8)
dls.show_batch(max_n=2)

,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>By now you've probably heard a bit about the new Disney dub of Miyazaki's classic film, Laputa: Castle In The Sky. During late summer of 1998, Disney released ""Kiki's Delivery Service"" on video which included a preview of the Laputa dub saying it was due out in ""1999"". It's obviously way past that year now, but the dub has been finally completed. And it's not ""Laputa: Castle In The Sky"", just ""Castle In The Sky"" for the dub, since Laputa is not such a nice word in Spanish (even though they use the word Laputa many times throughout the dub). You've also probably heard that world renowned composer, Joe Hisaishi, who scored the movie originally, went back to rescore the excellent music with new arrangements. Laputa came out before My Neighbor Totoro and after Nausicaa of the Valley of the Wind, which began Studio Ghibli and it's long string of",pos


In [ ]:
#| eval: false
dls_lm = TextDataLoadersNew.from_folder(path, is_lm=True, valid_pct=0.1, seed=42, folders=['train', 'test', 'unsup'], bs=64)
len(dls_lm.train_ds), len(dls_lm.valid_ds), len(dls_lm.vocab)

(90000, 10000, 50260)

## Export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()